# Notebook 1: InstructBLIP + Vicuna-7B — Single-Stage SFT on VizWiz-LF

**Architecture:** InstructBLIP with Vicuna-7B backbone  
**Strategy:** Freeze Vision Encoder + LLM. Train Q-Former ONLY (cast to FP32 for stable GradScaler).  
**Data:** `LF.json` (VizWiz Long-Form Rationale dataset) — synthetic answers only for training; expert answers held out for evaluation.  
**Environment:** Kaggle Dual-T4 (2×16 GB VRAM)

## 0. Install Dependencies

In [17]:
# NOTE: Do NOT upgrade Pillow — it breaks torchvision on Kaggle.
!pip install -q --upgrade \
    transformers \
    accelerate \
    datasets \
    huggingface_hub \
    evaluate \
    rouge_score \
    nltk \
    bert_score \
    sacrebleu \
    sentencepiece \
    bitsandbytes \
    opencv-python-headless

# METEOR requires NLTK data
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

## 1. Configuration

In [18]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
VIZWIZ_TRAIN_DIR   = "/kaggle/input/datasets/f230017abdullahkhan/viz-wiz-standard-with-lf/train/train"       # train images
VIZWIZ_VAL_DIR     = "/kaggle/input/datasets/f230017abdullahkhan/viz-wiz-standard-with-lf/val/val"           # val images
LF_JSON_PATH       = "/kaggle/input/datasets/f230017abdullahkhan/viz-wiz-standard-with-lf/LF.json"           # Long-Form rationale file
EXPERT_HOLDOUT_PATH = "/kaggle/working/expert_holdout.json"
OUTPUT_DIR         = "/kaggle/working/instructblip_sft"
HUB_REPO_ID        = "Abdullah-Khan-Niazi/instructblip-vizwiz-lf"   # <-- edit this

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID = "Salesforce/instructblip-vicuna-7b"

# ── Training hyper-params ──────────────────────────────────────────────────
BATCH_SIZE          = 4
GRAD_ACCUM          = 8          # effective batch = 32
NUM_EPOCHS          = 3
LR                  = 1e-4
WARMUP_STEPS        = 50        # replaces deprecated warmup_ratio
MAX_SEQ_LEN         = 512
NUM_WORKERS         = 4
SAVE_STEPS          = 50
SEED                = 42

# ── W&B ────────────────────────────────────────────────────────────────────
WANDB_PROJECT = "vizwiz-lf-instructblip"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config loaded.")

Config loaded.


## 2. Secrets & Logins

In [ ]:

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN   = ''

login(token=HF_TOKEN)
print("Logged in to HF Hub.")

Logged in to HF Hub.


## 3. High-Speed Image Path Indexer

> **CRITICAL:** Build a RAM dictionary once. Never call `os.path.exists()` inside `__getitem__`.

In [20]:
import glob

IMAGE_PATH_MAP: dict[str, str] = {}

for img_path in glob.glob(os.path.join(VIZWIZ_TRAIN_DIR, "*.jpg")):
    IMAGE_PATH_MAP[os.path.basename(img_path)] = img_path

for img_path in glob.glob(os.path.join(VIZWIZ_VAL_DIR, "*.jpg")):
    IMAGE_PATH_MAP[os.path.basename(img_path)] = img_path

print(f"Indexed {len(IMAGE_PATH_MAP):,} images into RAM.")

Indexed 31,704 images into RAM.


## 4. Parse `LF.json` — Split Synthetic vs Expert

In [21]:
import json
import os

with open(LF_JSON_PATH, "r", encoding="utf-8") as f:
    lf_data = json.load(f)

synthetic_train: list[dict] = []
expert_holdout:  list[dict] = []

for image_id, item in lf_data.items():
    # 1. Extract the actual filename from the end of the image_url
    image_url = item.get("image_url", "")
    image_filename = os.path.basename(image_url) if image_url else ""
    
    question = item.get("question", "").strip()
    long_answers = item.get("long_answers", {})

    # 2. Skip samples whose image is not in our RAM index
    if image_filename not in IMAGE_PATH_MAP:
        continue

    # 3. Parse the answers
    for source, answer_data in long_answers.items():
        if isinstance(answer_data, dict):
            answer_text = answer_data.get("answer_paragraph", "")
        else:
            answer_text = str(answer_data)
            
        if not answer_text.strip():
            continue

        record = {
            "image_id":       image_id,
            "image_filename": image_filename,
            "question":       question,
            "answer":         answer_text.strip(),
            "source":         source,
        }
        
        # Route expert / human answers to holdout; everything else to training
        if "expert" in source.lower() or "human" in source.lower():
            expert_holdout.append(record)
        else:
            synthetic_train.append(record)

# Persist expert holdout immediately
with open(EXPERT_HOLDOUT_PATH, "w", encoding="utf-8") as f:
    json.dump(expert_holdout, f, indent=2, ensure_ascii=False)

print(f"✅ Synthetic train samples : {len(synthetic_train):,}")
print(f"✅ Expert holdout samples  : {len(expert_holdout):,}")
print(f"✅ Expert holdout saved to : {EXPERT_HOLDOUT_PATH}")

✅ Synthetic train samples : 3,596
✅ Expert holdout samples  : 600
✅ Expert holdout saved to : /kaggle/working/expert_holdout.json


## 5. Load Processor & Model (Freeze Everything Except Q-Former)

In [22]:
import torch
from transformers import (
    InstructBlipProcessor, 
    InstructBlipForConditionalGeneration,
    BitsAndBytesConfig
)
from peft import prepare_model_for_kbit_training

print("Loading processor...")
# Add use_fast=False to avoid the ModuleNotFoundError
processor = InstructBlipProcessor.from_pretrained(MODEL_ID, use_fast=False)

print("Loading model in 8-bit to fit dual-T4 VRAM...")
# ... rest of your Cell 5 code
# ── 1. 8-Bit Config (Skip trainable layers) ────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_skip_modules=["qformer", "language_projection"] 
)

model = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# ── 2. CRITICAL OOM FIXES ──────────────────────────────────────────────────
# Prepare the 8-bit model for training
model = prepare_model_for_kbit_training(model)

# Manually enable gradient checkpointing to save massive amounts of VRAM
model.gradient_checkpointing_enable()

# ── 3. Freeze & Unfreeze ───────────────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = False

# Unfreeze Q-Former
for param in model.qformer.parameters():
    param.requires_grad = True

# Unfreeze Language Projection
for param in model.language_projection.parameters():
    param.requires_grad = True

# Explicitly cast trainable params to FP32 for stability
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Loading processor...
Loading model in 8-bit to fit dual-T4 VRAM...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(


Trainable params: 188,809,984 / 7,913,726,720 (2.39%)


## 6. PyTorch Dataset

In [23]:
import cv2
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

class VizWizLFDataset(Dataset):
    """
    Loads VizWiz Long-Form samples for InstructBLIP.

    Image loading policy:
      - Use cv2.imread + cv2.resize(224, 224) for speed and to prevent VRAM spikes.
      - Look up paths from IMAGE_PATH_MAP (never os.path.exists inside __getitem__).
    """
    def __init__(self, records: list[dict], processor, max_seq_len: int = 512):
        self.records     = records
        self.processor   = processor
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]

        # ── Fast image load via cv2 (BGR→RGB, resize to 224) ──────────────
        img_path = IMAGE_PATH_MAP[rec["image_filename"]]
        bgr = cv2.imread(img_path)
        if bgr is None:
            # Fallback: black frame (should never happen after path-map indexing)
            bgr = np.zeros((224, 224, 3), dtype=np.uint8)
        rgb = cv2.cvtColor(cv2.resize(bgr, (224, 224)), cv2.COLOR_BGR2RGB)
        image = Image.fromarray(rgb)

        question = rec["question"]
        answer   = rec["answer"]

        # ── Processor: image + question ────────────────────────────────────
        encoding = self.processor(
            images=image,
            text=question,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_seq_len,
            padding="max_length",
        )

        # ── Labels: tokenise the answer ────────────────────────────────────
        label_enc = self.processor.tokenizer(
            answer,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_seq_len,
            padding="max_length",
        )
        labels = label_enc["input_ids"].squeeze(0).clone()
        # Mask padding tokens from loss
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values":   encoding["pixel_values"].squeeze(0),
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "qformer_input_ids":      encoding.get("qformer_input_ids",
                                        torch.zeros(32, dtype=torch.long)).squeeze(0),
            "qformer_attention_mask": encoding.get("qformer_attention_mask",
                                        torch.ones(32, dtype=torch.long)).squeeze(0),
            "labels":         labels,
        }

## 7. DataLoaders

In [24]:
import random
from torch.utils.data import DataLoader

random.seed(SEED)
random.shuffle(synthetic_train)

# 90/10 train/val split within synthetic data
split_idx   = int(0.9 * len(synthetic_train))
train_records = synthetic_train[:split_idx]
val_records   = synthetic_train[split_idx:]

train_dataset = VizWizLFDataset(train_records, processor, MAX_SEQ_LEN)
val_dataset   = VizWizLFDataset(val_records,   processor, MAX_SEQ_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")

Train batches : 809
Val batches   : 90


## 8. Optimizer, Scheduler, GradScaler

> Uses **modern AMP API**: `torch.amp.GradScaler('cuda')` and `torch.amp.autocast('cuda')` — replaces deprecated `torch.cuda.amp` forms.

In [25]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR, ConstantLR

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)

total_steps   = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
warmup_steps  = WARMUP_STEPS  # explicit steps, not ratio

warmup_sched  = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0,
                          total_iters=warmup_steps)
decay_sched   = LinearLR(optimizer, start_factor=1.0,  end_factor=0.0,
                          total_iters=max(total_steps - warmup_steps, 1))
scheduler     = SequentialLR(optimizer,
                              schedulers=[warmup_sched, decay_sched],
                              milestones=[warmup_steps])

# Modern GradScaler API (PyTorch ≥ 2.0)
scaler = torch.amp.GradScaler('cuda')

print(f"Total optimiser steps : {total_steps}")

Total optimiser steps : 303


## 9. Kaggle Disk-Saver Helper

> Deletes the **previous** checkpoint before saving the new one to stay within the 19.5 GB limit.

In [31]:
import shutil
import glob
import torch
import json
from pathlib import Path

class KaggleDiskSaver:
    """Bypasses buggy transformers save_pretrained logic to save ONLY trainable weights."""
    def __init__(self, base_dir: str):
        self.base_dir = Path(base_dir)

    def save(self, model, processor, step: int, loss: float) -> Path:
        new_ckpt = self.base_dir / f"checkpoint-step{step}-loss{loss:.4f}"
        
        # 1. Aggressive Cleanup: Delete ANY existing checkpoint folders
        old_checkpoints = glob.glob(str(self.base_dir / "checkpoint-step*"))
        for old_path in old_checkpoints:
            shutil.rmtree(old_path, ignore_errors=True)
            print(f"  [DiskSaver] Cleaned up: {Path(old_path).name}")

        new_ckpt.mkdir(parents=True, exist_ok=True)

        # 2. DIRECT SAVE: Bypass model.save_pretrained()
        # Extract only weights that have requires_grad=True (Q-Former + Proj)
        trainable_state_dict = {n: p.cpu() for n, p in model.named_parameters() if p.requires_grad}
        
        # Save weights directly as a standard PyTorch bin file
        torch.save(trainable_state_dict, new_ckpt / "pytorch_model.bin")
        
        # Manually save configurations so from_pretrained knows what this model is
        model.config.save_pretrained(new_ckpt)
        if hasattr(model, "generation_config"):
            model.generation_config.save_pretrained(new_ckpt)
            
        # Processor save usually doesn't trigger the weight bug
        processor.save_pretrained(new_ckpt)
        
        print(f"  [DiskSaver] SUCCESS: Saved trainable weights to {new_ckpt.name}")
        return new_ckpt

disk_saver = KaggleDiskSaver(OUTPUT_DIR)

## 10. HuggingFace Hub Push Helper

In [33]:
from huggingface_hub import HfApi, create_repo

def hub_push(ckpt_path, epoch, metrics):
    hf_api = HfApi()
    
    # ── NEW: Ensure the repository exists before uploading ─────────
    try:
        create_repo(repo_id=HUB_REPO_ID, repo_type="model", exist_ok=True)
    except Exception as e:
        print(f"⚠️ Note: Could not verify/create repo: {e}")
    # ──────────────────────────────────────────────────────────────

    display_loss = metrics.get("val_loss") or metrics.get("train_loss") or 0.0
    loss_str = f"{display_loss:.4f}" if isinstance(display_loss, (int, float)) else str(display_loss)

    print(f"☁️ Pushing {ckpt_path.name} to Hugging Face Hub...")
    try:
        hf_api.upload_folder(
            folder_path=str(ckpt_path),
            repo_id=HUB_REPO_ID,
            path_in_repo=f"epoch_{epoch}_step_{global_step}_loss_{loss_str}",
            commit_message=f"End of Epoch {epoch} - Loss {loss_str}",
        )
        print("✅ Hub upload successful!")
    except Exception as e:
        print(f"❌ Hub upload failed: {e}")

## 11. Training Loop — Single-Stage SFT

In [37]:
import gc
import torch

# Delete model/optimizer if they exist in the current namespace
if 'model' in locals(): del model
if 'optimizer' in locals(): del optimizer

gc.collect()
torch.cuda.empty_cache()
# Set this flag to help with memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("VRAM Purged.")

VRAM Purged.


In [34]:
import csv
import os
import torch
import glob
import re
import json
from tqdm.auto import tqdm
from transformers import InstructBlipForConditionalGeneration
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR

# 1. ── Checkpoint Detection & Weight Loading ──────────────────────────────
def get_latest_checkpoint(base_dir):
    ckpts = glob.glob(os.path.join(base_dir, "checkpoint-step*"))
    if not ckpts: return None, 0
    def extract_step(path):
        match = re.search(r"checkpoint-step(\d+)", path)
        return int(match.group(1)) if match else 0
    latest_path = max(ckpts, key=extract_step)
    return latest_path, extract_step(latest_path)

ckpt_path, detected_step = get_latest_checkpoint(OUTPUT_DIR)

if ckpt_path:
    print(f"🔄 Auto-Resume: Found latest checkpoint at Step {detected_step}")
    # Load model if not already in memory
    if 'model' not in locals():
        print(f"📥 Loading weights from {ckpt_path}...")
        model = InstructBlipForConditionalGeneration.from_pretrained(
            ckpt_path,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
    global_step = detected_step
else:
    print("🆕 No checkpoint found. Starting from Step 0.")
    global_step = 0

# 2. ── Re-Initialize Training Tools (CRITICAL FIX) ────────────────────────
# Re-apply training settings to the model
model.gradient_checkpointing_enable()
for param in model.parameters(): param.requires_grad = False
for param in model.qformer.parameters(): param.requires_grad = True
for param in model.language_projection.parameters(): param.requires_grad = True

# Explicitly cast trainable params to FP32 for stability
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Re-create Optimizer
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)

# Re-create Scheduler[cite: 1]
total_steps_in_run = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
warmup_sched = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=WARMUP_STEPS)
decay_sched  = LinearLR(optimizer, start_factor=1.0,  end_factor=0.0, 
                         total_iters=max(total_steps_in_run - WARMUP_STEPS, 1))
scheduler    = SequentialLR(optimizer, schedulers=[warmup_sched, decay_sched], milestones=[WARMUP_STEPS])

# Catch up the scheduler to where we left off
for _ in range(global_step):
    scheduler.step()

# Re-create GradScaler[cite: 1]
scaler = torch.amp.GradScaler('cuda')

# Determine primary device[cite: 1]
primary_device = next((p.device for p in model.qformer.parameters() if p.requires_grad), torch.device("cuda:0"))

# 3. ── Setup CSV Logging ──────────────────────────────────────────────────
loss_log_path = os.path.join(OUTPUT_DIR, "loss_log.csv")
if not os.path.exists(loss_log_path):
    with open(loss_log_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "val_loss"])

# 4. ── Training Loop ──────────────────────────────────────────────────────
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    running_loss, train_steps = 0.0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]")
    for step, batch in enumerate(pbar, 1):
        # Skip batches if we are resuming mid-epoch
        # (Optional: ensures we don't repeat the exact same images)
        if epoch == 1 and step <= (global_step * GRAD_ACCUM):
            continue

        batch = {k: v.to(primary_device) if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}

        with torch.amp.autocast('cuda'):
            outputs = model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                qformer_input_ids=batch["qformer_input_ids"],
                qformer_attention_mask=batch["qformer_attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % SAVE_STEPS == 0:
                current_train_loss = running_loss / max(train_steps, 1)
                new_ckpt = disk_saver.save(model, processor, global_step, current_train_loss)
                hub_push(new_ckpt, epoch, {"train_loss": current_train_loss})
                model.train()

        running_loss += loss.item() * GRAD_ACCUM
        train_steps  += 1
        pbar.set_postfix({"loss": f"{running_loss/train_steps:.4f}", "step": global_step})

    # End of Epoch Validation
    model.eval()
    val_loss_sum, val_steps = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]"):
            batch = {k: v.to(primary_device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            with torch.amp.autocast('cuda'):
                outputs = model(**batch)
            val_loss_sum += outputs.loss.item()
            val_steps += 1
    
    val_loss = val_loss_sum / val_steps
    print(f"\nEpoch {epoch}: train_loss={running_loss/train_steps:.4f} val_loss={val_loss:.4f}")
    
    with open(loss_log_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, running_loss/train_steps, val_loss])

    # Final Epoch Save
    final_ckpt = disk_saver.save(model, processor, global_step, val_loss)
    hub_push(final_ckpt, epoch, {"val_loss": val_loss})

print("\n✅ Training complete.")

🔄 Auto-Resume: Found latest checkpoint at Step 101


Epoch 1/3 [train]:   0%|          | 0/809 [00:00<?, ?it/s]

Epoch 1/3 [val]:   0%|          | 0/90 [00:00<?, ?it/s]


Epoch 1: train_loss=6.0125 val_loss=5.9734
  [DiskSaver] Cleaned up: checkpoint-step101-loss5.9734
  [DiskSaver] SUCCESS: Saved trainable weights to checkpoint-step101-loss5.9734
☁️ Pushing checkpoint-step101-loss5.9734 to Hugging Face Hub...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Epoch 3/3 [val]:   0%|          | 0/90 [00:00<?, ?it/s]


Epoch 3: train_loss=5.5782 val_loss=5.7445
  [DiskSaver] Cleaned up: checkpoint-step300-loss5.5790
  [DiskSaver] SUCCESS: Saved trainable weights to checkpoint-step303-loss5.7445
☁️ Pushing checkpoint-step303-loss5.7445 to Hugging Face Hub...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Hub upload successful!

✅ Training complete.


## 12. Final Hub Upload — All Artefacts

In [39]:
# Push all evaluation artefacts to Hub
for fname in ["expert_holdout.json", "expert_predictions.json",
              "auto_metrics.json",   "llm_judge_metrics.json",
              "llm_judge_raw.json",  "loss_log.csv"]:
    fpath = os.path.join(
        "/kaggle/working" if fname == "expert_holdout.json" else OUTPUT_DIR,
        fname
    )
    if os.path.exists(fpath):
        hf_api.upload_file(
            path_or_fileobj=fpath,
            path_in_repo=f"artefacts/{fname}",
            repo_id=HUB_REPO_ID,
            repo_type="model",
            commit_message=f"Upload artefact: {fname}",
        )
        print(f"  Uploaded: {fname}")

print("\n✅ All artefacts pushed to HF Hub.")

  Uploaded: expert_holdout.json
  Uploaded: expert_predictions.json
  Uploaded: auto_metrics.json
  Uploaded: loss_log.csv

✅ All artefacts pushed to HF Hub.
